# Atrial Fibrillation Ablation: Late Relapse Classification

Binary classification study predicting **late relapse** (3–12 months post-ablation) in non-diabetic atrial fibrillation patients.

**Key aspects:**
- Small, imbalanced dataset (~114 patients, 2.8:1 class ratio)
- 120+ clinical biomarkers (pre/post-ablation blood work, echocardiography, inflammatory markers)
- Systematic evaluation of class-imbalance mitigation strategies (SMOTE-Tomek, Random Under-Sampling)
- Mutual Information feature selection (k = 15, 17, 20) combined with Random Forest and XGBoost classifiers
- Hyperparameter optimisation via GridSearchCV with Repeated Stratified K-Fold cross-validation

> **Note:** The original clinical dataset is private. This notebook uses synthetic data generated to match the original schema, class distribution, and statistical properties.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from collections import Counter

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE, SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import (
    StratifiedKFold, RepeatedStratifiedKFold,
    GridSearchCV, cross_val_score, cross_validate,
    train_test_split
)
from sklearn.metrics import (
    make_scorer, f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score, balanced_accuracy_score,
    classification_report
)

from xgboost import XGBClassifier, plot_importance

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.combine import SMOTETomek

In [ ]:
def imbalanced_ratio(y):
    counter = Counter(y)
    # Get the number of instances in the majority and minority classes
    majority_class = max(counter, key=counter.get)
    minority_class = min(counter, key=counter.get)
    majority_count = counter[majority_class]
    minority_count = counter[minority_class]
    # Calculate the imbalance ratio
    imbalance_ratio = majority_count / minority_count
    print(f'Counts: {counter}')
    print(f'Imbalance Ratio: {imbalance_ratio:.2f}')

def drop_high_correlations(df, thresh=0.95):
    corr_matrix = df.corr().abs()

    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > thresh)]

    return df.drop(to_drop, axis=1)

In [ ]:
data = pd.read_csv('data/synthetic_data.csv')
dictionary = pd.read_csv('data/clean_dictionary_non_diabetic.csv')

data.info()

X = data.drop(columns=['late_relapse'])
X = X.astype('float64')
y = data['late_relapse']

X.head()

In [ ]:
# COMPUTING THE IMBALANCE RATIO
imbalanced_ratio(y)

In [ ]:
for i in range(X.shape[1]):  # Loop through each column
    col_name = X.columns[i]  # Get column name
    if dictionary.loc[dictionary['variable_name'] == col_name, 'variable_type'].values[0] == 'numerical':
        X[col_name] = X[col_name].astype('float64')
    else:
        X[col_name] = X[col_name].astype('int64')

# Identify numeric columns
numeric_cols = X.select_dtypes(include=['float64']).columns

# Apply StandardScaler only to numeric columns
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

print(X.head())
print(X.shape)

In [ ]:
# Drop high correlated features
X = drop_high_correlations(X)
print(X.shape)

In [ ]:
# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Define feature selection techniques, classifiers, and oversampling methods
feature_selection_techniques = ['MutualInfo']
n_features = [15,17,20]
unbalanced_techniques = ['None','SMOTE-TOMEK','RandomUnderSampler']
classifiers = ['RF','GB']

# Cross-validation strategy
skf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
measure_cv = make_scorer(roc_auc_score, greater_is_better=True)

# DataFrame to store results
MLA_columns = [
    'MLA Name', 'Accuracy_train_mean', 'Accuracy_train_sd', 'Accuracy_test', 'Precision_train_mean', 'Precision_train_sd', 'Precision_test',
    'Recall_train_mean', 'Recall_train_sd', 'Recall_test', 'F1_train_mean', 'F1_train_sd', 'F1_test', 'AUC_train_mean', 'AUC_train_sd', 'AUC_test',
    'Balanced Accuracy_train_mean', 'Balanced Accuracy_train_sd', 'Balanced Accuracy_test'
]
MLA_compare = pd.DataFrame(columns=MLA_columns)
row_index = 0

# Loop through oversampling techniques
for technique in unbalanced_techniques:
    # Define the oversampling step
    if technique == 'RandomUnderSampler':
        sampler = RandomUnderSampler(random_state=42)
    elif technique == 'SMOTE':
        sampler = SMOTE(random_state=42)
    elif technique == 'SMOTE-TOMEK':
        sampler = SMOTETomek(random_state=42)
    elif technique == 'ADASYN':
        sampler = ADASYN(random_state=42)
    elif technique == 'BorderlineSMOTE':
        sampler = BorderlineSMOTE(random_state=42)
    else:
        sampler = None  # No oversampling

    # Loop through feature selection techniques
    for feature_technique in feature_selection_techniques:
        for n_feat in n_features:
            # Define the feature selection step
            if feature_technique == 'RFE':
                selector = RFE(estimator=LogisticRegression(penalty='l1', solver='liblinear', random_state=42), n_features_to_select=n_feat, step=1)
            elif feature_technique == 'MutualInfo':
                selector = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=42), k=n_feat)
            elif feature_technique == 'PCA':
                selector = PCA(n_components=n_feat)
            else:
                selector = None  # No feature selection

            # Loop through classifiers
            for classifier in classifiers:
                # Define the classifier and its hyperparameter grid
                if classifier == 'LR':
                    model = LogisticRegression(solver='liblinear', random_state=42)
                    param_grid = {
                        'classifier__penalty': ['l1', 'l2'],
                        'classifier__C': [0.01, 0.1, 1, 10, 100],
                        'classifier__class_weight': [None, 'balanced']
                    }
                elif classifier == 'RF':
                    model = RandomForestClassifier(random_state=42)
                    param_grid = {
                        'classifier__n_estimators': [100, 300, 500],
                        'classifier__max_depth': [3, 6, 9],
                        'classifier__class_weight': [None, 'balanced']
                    }
                elif classifier == 'GB':
                    model = XGBClassifier(random_state=42)
                    param_grid = {
                        'classifier__learning_rate': [0.01, 0.05, 0.1],
                        'classifier__n_estimators': [100, 200, 300],
                        'classifier__max_depth': [3, 5, 7],
                        'classifier__scale_pos_weight': [1, 3, 5]
                    }
                elif classifier == 'SVM':
                    model = SVC(probability=True, random_state=42)
                    param_grid = {
                        'classifier__C': [0.1, 1, 10],
                        'classifier__kernel': ['rbf', 'sigmoid'],
                        'classifier__gamma': ['scale', 'auto'],
                        'classifier__class_weight': [None, 'balanced']
                    }

                # Create the pipeline
                steps = []
                if sampler is not None:
                    steps.append(('sampler', sampler))
                if selector is not None:
                    steps.append(('selector', selector))
                steps.append(('classifier', model))
                pipeline = Pipeline(steps)

                # Perform GridSearchCV with the pipeline
                grid_search = GridSearchCV(pipeline, param_grid, scoring=measure_cv, cv=skf, verbose=1, n_jobs=-1)
                grid_search.fit(X_train, y_train)

                # Get the best model and evaluate it
                best_model = grid_search.best_estimator_
                predicted = best_model.predict(X_test)
                predicted_proba = best_model.predict_proba(X_test)[:, 1]
                
                cv_accuracy = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='accuracy')
                cv_precision = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='precision')
                cv_recall = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='recall')
                cv_f1 = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='f1')
                cv_auc = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='roc_auc')
                cv_baccuracy = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='balanced_accuracy')
                
                # Calculate metrics
                MLA_name = f"{classifier}_{feature_technique}_{n_feat}_{technique}"
                MLA_compare.loc[row_index, 'MLA Name'] = MLA_name
                MLA_compare.loc[row_index, 'Accuracy_train_mean'] = cv_accuracy.mean()
                MLA_compare.loc[row_index, 'Accuracy_train_sd'] = cv_accuracy.std()
                MLA_compare.loc[row_index, 'Accuracy_test'] = accuracy_score(y_test, predicted)
                MLA_compare.loc[row_index, 'Precision_train_mean'] = cv_precision.mean()
                MLA_compare.loc[row_index, 'Precision_train_sd'] = cv_precision.std()
                MLA_compare.loc[row_index, 'Precision_test'] = precision_score(y_test, predicted)
                MLA_compare.loc[row_index, 'Recall_train_mean'] = cv_recall.mean()
                MLA_compare.loc[row_index, 'Recall_train_sd'] = cv_recall.std()
                MLA_compare.loc[row_index, 'Recall_test'] = recall_score(y_test, predicted)
                MLA_compare.loc[row_index, 'F1_train_mean'] = cv_f1.mean()
                MLA_compare.loc[row_index, 'F1_train_sd'] = cv_f1.std()
                MLA_compare.loc[row_index, 'F1_test'] = f1_score(y_test, predicted)
                MLA_compare.loc[row_index, 'AUC_train_mean'] = cv_auc.mean()
                MLA_compare.loc[row_index, 'AUC_train_sd'] = cv_auc.std()
                MLA_compare.loc[row_index, 'AUC_test'] = roc_auc_score(y_test, predicted_proba)
                MLA_compare.loc[row_index, 'Balanced Accuracy_train_mean'] = cv_baccuracy.mean()
                MLA_compare.loc[row_index, 'Balanced Accuracy_train_sd'] = cv_baccuracy.std()
                MLA_compare.loc[row_index, 'Balanced Accuracy_test'] = balanced_accuracy_score(y_test, predicted)
                row_index += 1

                # Save results
                MLA_compare.sort_values(by=['AUC_train_mean'], ascending=False, inplace=True)
                print(MLA_compare.head())
                MLA_compare.to_csv('results.csv', index=False)

            if feature_technique == 'None':
                break

C:\Users\Raquel\PycharmProjects\ablation_class\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                MLA Name Accuracy_train_mean Accuracy_train_sd Accuracy_test  \
1  GB_MutualInfo_15_None            0.654795          0.081939       0.73913   
0  RF_MutualInfo_15_None            0.726374          0.033528       0.73913   

  Precision_train_mean Precision_train_sd Precision_test Recall_train_mean  \
1             0.249048           0.278263            0.5             0.157   
0                 0.04           0.195959            0.0             0.008   

  Recall_train_sd Recall_test F1_train_mean F1_train_sd F1_test  \
1        0.153626    0.166667       0.18156    0.175419    0.25   
0        0.039192         0.0      0.013333     0.06532     0.0   

  AUC_train_mean AUC_train_sd  AUC_test Balanced Accuracy_train_mean  \
1       0.472368     0.152064  0.696078                     0.495753   
0       0.466154     0.137897  0.558824                     0.495868   

  Balanced Accuracy_train_sd Balanced Accuracy_test  
1                   0.090928               0.553922

C:\Users\Raquel\PycharmProjects\ablation_class\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Raquel\PycharmProjects\ablation_class\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Raquel\PycharmProjects\ablation_class\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()

In [ ]:
# ---- COMPARE MODELS ---- #

feature_selection_techniques = ['MutualInfo'] # 'PCA'
n_features = [15]
unbalanced_techniques = ['RandomUnderSampler']
classifiers = ['RF'] # 'LR','RF','GB', 'T'
#skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
skf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
measure_cv=make_scorer(roc_auc_score, greater_is_better=True)

for technique in unbalanced_techniques:
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
    if technique == 'RandomUnderSampler':
        rus = RandomUnderSampler(random_state=42)
        imbalanced_ratio(y_train)
        X_train, y_train = rus.fit_resample(X_train,y_train)
        imbalanced_ratio(y_train)
    if technique == 'SMOTE':
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        imbalanced_ratio(y_train)
    if technique == 'SMOTE-TOMEK':
        smote_tomek = SMOTETomek(random_state=42)
        X_train, y_train = smote_tomek.fit_resample(X_train, y_train)
        imbalanced_ratio(y_train)
    if technique == 'ADASYN':
        adasyn = ADASYN(random_state=42)
        X_train, y_train = adasyn.fit_resample(X_train, y_train)
        imbalanced_ratio(y_train)
    if technique == 'BorderlineSMOTE':
        bsmote = BorderlineSMOTE(random_state=42)
        X_train, y_train = bsmote.fit_resample(X_train, y_train)
        imbalanced_ratio(y_train)
    for feature_technique in feature_selection_techniques:
        for n_feat in n_features:
            if feature_technique == 'RFE':
                model_selected = LogisticRegression(penalty='l1', solver='liblinear', random_state=42)
                rfe = RFE(estimator=model_selected, n_features_to_select=n_feat, step=1)
                rfe = rfe.fit(X_train, y_train)
                selected_features = X_train.columns[rfe.support_]
                X_train_sel = X_train[selected_features]
                X_test_sel = X_test[selected_features]       
            if feature_technique == 'MutualInfo':
                mi_selector = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=42), k=n_feat)
                X_kbest = mi_selector.fit_transform(X_train, y_train)
                # Print selected features
                selected_features = X_train.columns[mi_selector.get_support()]
                X_train_sel = X_train[selected_features]
                X_test_sel = X_test[selected_features]
            if feature_technique == 'PCA':
                pca = PCA(n_components=n_feat)
                X_train_sel = pca.fit_transform(X_train)
                X_test_sel = pca.transform(X_test)
                
            if feature_technique == 'None':
                X_train_sel = X_train
                X_test_sel = X_test
            for classifier in classifiers:
                if classifier == 'LR':
                    lr_model = LogisticRegression(solver='liblinear', random_state=42)
                    lr_param_grid = {
                        'penalty': ['l1', 'l2'],  # Regularization type
                        'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength
                        'class_weight': [None, 'balanced']  # Handling imbalance
                    }
                    lr_grid_search = GridSearchCV(estimator=lr_model, param_grid=lr_param_grid,scoring=measure_cv, cv=skf, verbose=1, n_jobs=-1)
                    lr_grid_search.fit(X_train_sel, y_train)
                    model = lr_grid_search.best_estimator_
                    
                if classifier == 'RF': 
                    rf_model = RandomForestClassifier(random_state=42)
                    rf_param_grid = {
                        'n_estimators': [100, 300, 500],  # Number of trees in the forest
                        'max_depth': [3, 6, 9],  # Maximum depth of the tree
                        #'min_samples_split': [2, 5, 10],  # Minimum samples required to split a node
                        #'min_samples_leaf': [1, 2, 4],  # Minimum samples required at a leaf node
                        #'max_features': ['auto', 'sqrt'],  # Number of features to consider for splitting
                        #'bootstrap': [True, False],  # Whether to use bootstrap sampling
                        'class_weight': [None, 'balanced']  # Weights for imbalanced classes
                    }
                    rf_grid_search = GridSearchCV(estimator=rf_model, param_grid=rf_param_grid,scoring=measure_cv, cv=skf, verbose=1, n_jobs=-1)
                    rf_grid_search.fit(X_train_sel, y_train)
                    model = rf_grid_search.best_estimator_
                    
                if classifier == 'GB': 
                    xgb_model = XGBClassifier(random_state=42)
                    xgb_param_grid = {
                        'learning_rate': [0.01, 0.05, 0.1],  # Step size for boosting
                        'n_estimators': [100, 200, 300],      # Number of boosting rounds
                        'max_depth': [3, 5, 7],              # Maximum depth of trees
                        #'subsample': [0.6, 0.8, 1.0],    # Fraction of samples used for training
                        #'colsample_bytree': [0.6, 0.8, 1.0],  # Fraction of features used for training
                        #'gamma': [0, 0.2, 0.4],          # Minimum loss reduction for a split
                        #'min_child_weight': [1, 2, 3],          # Minimum sum of instance weights in a child
                        #'reg_lambda': [0, 0.1, 1],             # L2 regularization
                        #'reg_alpha': [0, 0.1, 1],              # L1 regularization
                        'scale_pos_weight': [1, 3, 5]          # Adjust for imbalanced datasets
                    }
                    
                    xgb_grid_search = GridSearchCV(estimator=xgb_model, param_grid=xgb_param_grid,scoring=measure_cv, cv=skf, verbose=1, n_jobs=-1)
                    xgb_grid_search.fit(X_train_sel, y_train)
                    model = xgb_grid_search.best_estimator_
                    
                model = model.fit(X_train_sel, y_train)
                
                if classifier == 'LR':
                    coefficients = model.coef_[0]  # The coefficients for each feature
                    # If X_train is a DataFrame, use its column names directly
                    feature_names = X_train_sel.columns
                    
                    # Create a DataFrame to organize the coefficients and their absolute values
                    feature_importance_df = pd.DataFrame({
                        'Feature': feature_names,
                        'Coefficient': coefficients,
                        'Absolute Coefficient': np.abs(coefficients),
                        'Sign': ['positive' if coef > 0 else 'negative' for coef in coefficients]
                    })
                    
                    # Sort by absolute coefficient value (larger magnitude means more important feature)
                    feature_importance_df = feature_importance_df.sort_values(by='Absolute Coefficient', ascending=False)
                    
                    # Select top 20 most important features
                    top_20_features = feature_importance_df.head(20)
                    
                    # Define a color palette based on the sign of the coefficients
                    palette = {'positive': 'blue', 'negative': 'red'}
                    
                    # Plot the top 20 most important features
                    plt.figure(figsize=(10, 6))
                    sns.barplot(x='Absolute Coefficient', y='Feature', data=top_20_features, hue='Sign', dodge=False, palette=palette)
                    plt.title('Top 20 Feature Importance based on LogisticRegression Coefficients')
                    plt.xlabel('Absolute Coefficient Value')
                    plt.ylabel('Feature')
                    plt.legend(title='Sign')
                    plt.tight_layout()
                    plt.show()
                    
                    # Display top 20 feature importance
                    print("\nTop 20 Feature Importance based on Coefficients:")
                    print(top_20_features)
                if classifier == 'GB':
                    feature_importance = model.feature_importances_
                    feature_names = X_test_sel.columns
                    feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance})
                    # Sort features by importance in descending order
                    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
                    # Select top 20 most important features
                    top_20_features = feature_importance_df.head(20)
                    print(top_20_features)
                    #print(top_20_features)
                    # Plot feature importance using gain for the top 20 features
                    plot_importance(model, importance_type='gain', title='Feature Importance (Gain)', 
                                    show_values=False, max_num_features=20)
                    plt.show()
                if classifier == 'RF':
                    
                    # Extract feature importances
                    feature_importance = model.feature_importances_
                    feature_names = X_test_sel.columns  # Ensure X_test_sel is your feature matrix
                    
                    # Create a DataFrame for feature importances
                    feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance})
                    
                    # Sort features by importance in descending order
                    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
                    
                    # Select top 20 most important features
                    top_20_features = feature_importance_df.head(20)
                    
                    # Print the top 20 features
                    print(top_20_features)
                    
                    # Plot feature importance
                    plt.figure(figsize=(10, 8))
                    plt.barh(top_20_features['Feature'], top_20_features['Importance'], color='skyblue')
                    plt.xlabel('Importance')
                    plt.title('Top 20 Feature Importance (Random Forest)')
                    plt.gca().invert_yaxis()  # Invert y-axis to show the most important feature at the top
                    plt.show()
            if feature_technique == 'None':
               break 

## SHAP Explainability

SHAP (SHapley Additive exPlanations) values quantify the contribution of each feature to individual predictions.
Applied here to the best-performing configuration: **Random Forest + Mutual Information (k=15) + Random Under-Sampling**.

In [ ]:
# Reproduce best configuration: RF + MutualInfo k=15 + RandomUnderSampler
X_train_shap, X_test_shap, y_train_shap, y_test_shap = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

rus = RandomUnderSampler(random_state=42)
X_train_shap, y_train_shap = rus.fit_resample(X_train_shap, y_train_shap)

mi_sel = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=42), k=15)
X_train_shap = pd.DataFrame(
    mi_sel.fit_transform(X_train_shap, y_train_shap),
    columns=X_train_shap.columns[mi_sel.get_support()]
)
X_test_shap = X_test_shap[X_train_shap.columns]

rf_shap = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=42)
rf_shap.fit(X_train_shap, y_train_shap)

print("Selected features:", list(X_train_shap.columns))
print(classification_report(y_test_shap, rf_shap.predict(X_test_shap)))

In [ ]:
explainer = shap.TreeExplainer(rf_shap)
shap_vals = explainer.shap_values(X_test_shap)
# shap_vals is a list [class_0, class_1]; use class 1 (late relapse)

# Global feature importance — beeswarm
shap.summary_plot(shap_vals[1], X_test_shap, plot_type="dot", max_display=15)

# Single-prediction explanation — waterfall for first test sample
sv = shap.Explanation(
    values=shap_vals[1][0],
    base_values=explainer.expected_value[1],
    data=X_test_shap.iloc[0].values,
    feature_names=list(X_test_shap.columns)
)
shap.plots.waterfall(sv)